# Automotive Data Mapper - MVP: Data Validation Pipeline

**Date:** 2026-08-13  
**Author:** Luis Renteria Lezano  
[LinkedIn](https://www.linkedin.com/in/renteria-luis) | [GitHub](https://github.com/renteria-luis) | [Portfolio](https://luisrenteria.me)

## Executive Summary

* **Goal:** validate the raw automotive service-record feeds before mapping them into the canonical schema. This notebook determines which records the pipeline accepts, which it rejects, and why.
* **Input:** The raw feeds profiled in [`01_sources.ipynb`](01_sources.ipynb), representing three automotive data providers:
  * `shop_a`: CSV
  * `dealer_b`: XML
  * `fleet_c`: JSON
* **Expected input:** 97 records across the three feeds, including 10 records that are known to be impossible to map based on the source characteristics identified during profiling.
* **Data:** [`../data/raw/`](https://github.com/renteria-luis/automotive-data-mapper/tree/main/data/raw)
* **Data dictionary:** [`../docs/DATA_DICTIONARY.md`](https://github.com/renteria-luis/automotive-data-mapper/blob/main/docs/DATA_DICTIONARY.md)
* **Sample data documentation:** [`../docs/SAMPLE_DATA.md`](https://github.com/renteria-luis/automotive-data-mapper/blob/main/docs/SAMPLE_DATA.md)

This notebook turns the findings from `01_sources.ipynb` into measurable validation rules. The objective is not simply to report that the pipeline rejected 10 records, but to verify that it identified all 10 records that are impossible to map. This makes the validation result measurable: **10 known-invalid records exist, and the pipeline detects 10 of 10.**

In [ ]:
import re
from datetime import date, datetime, timezone

import pandas as pd
from pydantic import BaseModel, ConfigDict, ValidationError, field_validator

from src.mappings import DEALER_B_MAP, DEALER_B_UNMAPPED, FLEET_C_MAP, SHOP_A_MAP
from src.readers import read_dealer_b, read_fleet_c, read_shop_a

shop_a_raw = read_shop_a()
dealer_b_raw = read_dealer_b()
fleet_c_raw = read_fleet_c()

for name, df in [('shop_a', shop_a_raw), ('dealer_b', dealer_b_raw), ('fleet_c', fleet_c_raw)]:
    print(f'{name:10} {len(df):>3} records')

TOTAL = len(shop_a_raw) + len(dealer_b_raw) + len(fleet_c_raw)
print(f'{"total":10} {TOTAL:>3}')
assert TOTAL == 97

The three reads are now one line each. All the complexity from [`01_sources.ipynb`](01_sources.ipynb) CSV quotes, XML namespace, and JSON envelope; all of it is contained within [`../src/readers.py`](../src/readers.py), so we no longer need to think about it here.

That is what it means for the code to “survive” exploration.

## 1. The canonical schema

A single place defines what an acceptable record is. Everything else refers to it.

### 1.1 The decision: why Pydantic instead of pandas conditions

pandas can validate. `df[df['vin'].str.len() != 17]` returns VINs with the wrong length. It works.

The problem appears when you have two rules at the same time.

In [ ]:
demo = pd.DataFrame(
    {
        'record': ['A', 'B', 'C'],
        'vin': ['1FTFW1E50KFA1234', '1G1ZD5ST0LFO04821', '1HGCV1F30LA100234'],
        'description': ['oil change', None, 'brake pads'],
    }
)

bad_len = demo['vin'].str.len() != 17
bad_chars = demo['vin'].str.contains('[IOQ]')
bad_desc = demo['description'].isna()

demo.assign(bad_len=bad_len, bad_chars=bad_chars, bad_desc=bad_desc)

We would need one column for each condition, and even worse, for record B, what would the rejection code be? It fails two rules. 

With pandas, we would have to manually define which one takes priority, write the order somewhere, and maintain it when rule number eleven is added. The logic would be spread across the masks and the code that combines them.

> **Pydantic reverses the problem:** it validates one record at a time, stops at the first failure, and reports which field failed and why. This is exactly the structure of a reason code.

### 1.2 The ISO 3779 check digit

Of the three VIN checks, two are trivial: 17 characters and no `I`, `O`, or `Q`. The third is the only one that catches a typing error.

Position **9** of the VIN is not vehicle information. It is a number calculated from the other 16 characters. If someone transposes two characters, the calculation no longer matches -> [Official Algorithm Surce](https://www.ecfr.gov/current/title-49/subtitle-B/chapter-V/part-565/subpart-B/section-565.15)
$$\text{Check Digit} = \left( \sum_{i=1}^{17} (v_i \times p_i) \right) \pmod{11}$$
Where $v_i$ is the numerical value of the character at position $i$, y $p_i$ is the multiplier (weight) assigned to that position in the VIN. If the modulo result is 10, the check digit is the letter X.

In [ ]:
TRANSLIT = {
    **{c: i + 1 for i, c in enumerate("ABCDEFGH")},
    **{c: i + 1 for i, c in enumerate("JKLMN")},
    "P": 7,
    "R": 9,
    **{c: i + 2 for i, c in enumerate("STUVWXYZ")},
    **{str(d): d for d in range(10)},
}
WEIGHTS = [8, 7, 6, 5, 4, 3, 2, 10, 0, 9, 8, 7, 6, 5, 4, 3, 2]


def vin_check_digit(vin: str) -> str:
    """Position 9 of a VIN, computed from the other sixteen characters (ISO 3779)."""
    total = sum(TRANSLIT[c] * w for c, w in zip(vin, WEIGHTS))
    remainder = total % 11
    return "X" if remainder == 10 else str(remainder)

#### Testing function

[`SAMPLE_DATA.md`](../docs/SAMPLE_DATA.md) says that a `dealer_b` record has two adjacent characters transposed. It has 17 characters and contains no illegal characters, so it passes the first two checks.

In [ ]:
good = '1HGCV1F30LA100234'
transposed = good[:2] + good[3] + good[2] + good[4:]

for vin in (good, transposed):
    print(
        f'{vin}  length={len(vin)}  illegal={bool(set(vin) & set("IOQ"))}  '
        f'position 9={vin[8]}  calculated={vin_check_digit(vin)}'
    )

The check digit test must be run even if the other two checks pass, and they must be run in this order:

**length -> illegal characters -> check digit**

For example, if we ran the check digit test first on a 16-character VIN, the result would be meaningless and could even raise an error.

### 1.3 `VehicleEvent`, and each validator defines its own code

The fields come from [`../docs/DATA_DICTIONARY.md`](../docs/DATA_DICTIONARY.md). The decision we need to make here is **how the reason code gets from the validator to the rejection table**.

The solution: each validator puts its code at the beginning of the error message. Adding a new rule means adding a validator with its code in one place, without having to maintain a separate translation table.

Two details of the model:

* `extra='forbid'` makes an unknown field an error instead of letting it pass unnoticed. This is the opposite of silent behavior.
* `mode='before'` makes the validator run **before** Pydantic attempts to convert the value's type. Without it, an invalid date would fail with Pydantic's generic error message instead of the `E005` code.

In [ ]:
class VehicleEvent(BaseModel):
    """One service event on one vehicle, in the canonical shape."""

    # forbig: reject | ignore: ignores | allow: keeps
    model_config = ConfigDict(extra="forbid")

    source_id: str
    source_record_id: str
    vin: str
    vin_valid: bool | None = None
    event_date: date
    odometer_km: int | None = None
    odometer_source_unit: str | None = None
    raw_description: str
    normalized_description: str
    provider_name: str | None = None
    provider_city: str | None = None
    provider_province: str | None = None
    ingested_at: datetime  # pydantic not only validates but also converts

    @field_validator("vin", mode="before")
    @classmethod
    def check_vin(cls, v):
        if v is None or str(v).strip() == "":  # not empty
            raise ValueError("E004: vin is empty")
        v = str(v).strip().upper()             # convert to upper+str
        if len(v) != 17:                       # 17 chars
            raise ValueError(f"E001: vin has {len(v)} characters, expected 17")
        illegal = sorted(set(v) & set("IOQ"))
        if illegal:                           # not I, O, Q chars
            raise ValueError(f"E002: vin contains {illegal}, which are not used in VINs")
        expected = vin_check_digit(v)
        if v[8] != expected:                  # check digit
            raise ValueError(f"E003: check digit is {v[8]}, expected {expected}")
        return v

    @field_validator("event_date", mode="before")
    @classmethod
    def check_date(cls, v):
        if v is None or str(v).strip() == "":
            raise ValueError("E004: event_date is empty")
        try:
            parsed = date.fromisoformat(str(v))
        except ValueError:
            raise ValueError(f"E005: {v!r} is not a real calendar date")
        if parsed > date.today():
            raise ValueError(f"E006: {parsed.isoformat()} is in the future")
        return parsed

    @field_validator("source_record_id", "raw_description", mode="before")
    @classmethod
    def check_not_empty(cls, v, info):  # need info to know which field is validating
        if v is None or str(v).strip() == "":
            raise ValueError(f"E004: {info.field_name} is empty")
        return str(v).strip()

    @field_validator("odometer_km")  # mode: after, we need it to be converted to int first
    @classmethod
    def check_odometer(cls, v):
        if v is not None and v < 0:
            raise ValueError(f"E008: odometer is {v}, which cannot be negative")
        return v

### 1.4 `RejectedRecord`

A record that fails **is not an error; it is data**. That is the decision in DD-006, and it defines the structure of this table.

`raw_record` is the field that makes the difference. Without it, fixing a rejected record means manually searching for it in the original file. With it, the record travels together with its reason and can be corrected and reprocessed.

In [ ]:
class RejectedRecord(BaseModel):
    """One record the pipeline could not turn into a VehicleEvent."""

    source_id: str
    source_record_id: str | None = None
    reason_code: str
    detail: str
    raw_record: dict
    ingested_at: datetime

`source_record_id` must be None for `E010`: when an unmapped column is detected, because this error refers to the entire file and not to a specific record. `None` is also valid for a record that is so broken that its ID cannot be read. In that case, `raw_record` is what remains available for investigation, that is why this field is mandatory.

## 2. Mapping & Normalization

Here we apply what [`01_sources.ipynb`](01_sources.ipynb) defined. The mapping restructures and transforms; it does not judge if something is valid or not. All validation decisions belong in section 3.

This separation is deliberate: if mapping also rejected records, there would be two different places producing reason codes, and neither would have the complete list.

In [ ]:
def to_km(value, unit: str | None) -> int | None:
    """Odometer to whole kilometres. Blank, zero and unparseable all mean unknown."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    text = str(value).replace(",", "").strip()
    if text == "" or text.lower() == "nan":
        return None
    try:
        number = int(float(text))  # int("12000.0") would NOT pass
    except ValueError:
        return None
    if number == 0:
        return None
    return round(number * 1.609344) if unit == "mi" else number


def normalize(text) -> str | None:
    """Lowercased, whitespace collapsed. Input to the phase two taxonomy."""
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return None
    return re.sub(r"\s+", " ", str(text)).strip().lower()


PROVIDER_NAMES = {
    "riverside auto service": "Riverside Auto Service",
    "riverside auto service ltd": "Riverside Auto Service",
    "forest city motors": "Forest City Motors",
    "forest city motors ltd": "Forest City Motors",
}


def standardize_provider(name) -> str | None:
    """One business, one spelling. Three variants of each name arrive in the feeds."""
    if name is None or (isinstance(name, float) and pd.isna(name)):
        return None
    return PROVIDER_NAMES.get(normalize(name), str(name).strip())

**Two decisions are documented in `to_km`.**

A zero odometer reading is treated as unknown, not as 0 km. A vehicle with a service history is unlikely to have zero mileage, so that zero is almost certainly a field the shop left blank and the system filled with a default value.

And if a value cannot be converted, `None` is returned instead of raising an error. A missing odometer makes the record **incomplete**, not **invalid**, and [`SAMPLE_DATA.md`](../docs/SAMPLE_DATA.md) states this explicitly: rejecting it would be just as incorrect as accepting a broken VIN.

### 2.2 One mapper per feed

Each mapper applies its dictionary and the transformations justified by the profiling. Dates are normalized to ISO text here, but if they cannot be parsed, **the original value is passed through unchanged**: mapping does not make decisions; it only prepares the data.

In [ ]:
INGESTED_AT = datetime.now(timezone.utc)


def map_shop_a(row) -> dict:
    unit = str(row["ODOMETER_MEASURE"]).strip().lower() if pd.notna(row["ODOMETER_MEASURE"]) else None
    description = row["SERVICE_DESCRIPTION"] if pd.notna(row["SERVICE_DESCRIPTION"]) else None
    try:
        event_date = datetime.strptime(str(row["RO_OPEN_DATE"]).strip(), "%m/%d/%Y").date().isoformat()
    except ValueError:
        event_date = str(row["RO_OPEN_DATE"]).strip()

    return {
        "source_id": "shop_a",
        "source_record_id": str(row["RO_INVOICE_NUMBER"]).strip(),
        "vin": row["VIN"],
        "event_date": event_date,
        "odometer_km": to_km(row["MILEAGE"], unit),
        "odometer_source_unit": unit,
        "raw_description": str(description).strip() if description is not None else None,
        "normalized_description": normalize(description),
        "provider_name": standardize_provider(row["LOCATION_NAME"]),
        "provider_city": str(row["CITY"]).strip(),
        "provider_province": str(row["STATE"]).strip().upper(),
        "ingested_at": INGESTED_AT,
    }